# PHASE 2 — REGRESSION


# Day 13 — Ensemble Learning (Random Forests)


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Understand the concept of Ensemble Learning (Wisdom of the Crowd).
- Explain how Bagging (Bootstrap Aggregating) works.
- Implement a `RandomForestRegressor` to fix the overfitting problems of single Decision Trees.
- Tune the hyperparameters `n_estimators` and `max_depth`.


## 2. Prerequisites
- Day 12 (Decision Trees).


## 3. Concept
Yesterday, we learned that a single Decision Tree is wildly unstable. If you change even one row of training data, the entire structure of the tree might change, leading to massive Overfitting.

**Ensemble Learning** is the idea that combining multiple weak, unstable models creates one incredibly strong, stable model. It's the mathematical version of the "Wisdom of the Crowd."

A **Random Forest** is simply an ensemble of hundreds of Decision Trees.


## 4. Why Does This Matter?
Random Forest is widely considered the "Swiss Army Knife" of Machine Learning. If you are handed a tabular dataset and only have 5 minutes to get a decent prediction, you run a Random Forest. It is incredibly robust to outliers, requires no scaling, handles non-linear data, and doesn't overfit easily.


## 5. Intuition
Imagine you want to guess the exact number of jellybeans in a massive jar. 
- A single person (Decision Tree) will likely be wildly wrong (high variance).
- If you ask 1,000 random people, some will guess way too high, and some way too low. 
- But if you **average** all 1,000 guesses, the final answer will be shockingly close to the truth.

A Random Forest trains 1,000 Decision Trees and simply takes the mathematical average of all their predictions.


## 6. Mathematical Foundation (Bagging)
If all 100 trees in the forest saw the exact same data, they would all build the exact same tree, rendering the average useless. 

Random Forests use a mathematical trick called **Bagging (Bootstrap Aggregating)**:
1. **Bootstrap**: The algorithm randomly samples rows from your dataset *with replacement* to create 100 slightly different, "fake" datasets.
2. **Feature Randomness**: At every split, each tree is only allowed to look at a random subset of features (e.g., it can only choose between Age or Salary, but not Education).
3. **Aggregate**: Train 100 trees on these random subsets, then average their predictions.

This guarantees every tree is fundamentally unique, making their average incredibly powerful and resistant to overfitting.


## 7. Scikit-learn API
```python
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
```
`n_estimators` is the number of trees in the forest. Default is 100.


## 8. Simple Example
Let's generate a highly noisy, non-linear dataset. We will compare a single Decision Tree against a Random Forest of 100 trees.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# 1. Generate noisy sine wave data
np.random.seed(42)
X = np.sort(5 * np.random.rand(80, 1), axis=0)
y = np.sin(X).ravel() + np.random.randn(80) * 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Train a single Unconstrained Tree
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

# 3. Train a Random Forest (100 trees)
forest = RandomForestRegressor(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

print('Single Tree RMSE:  ', np.sqrt(mean_squared_error(y_test, tree.predict(X_test))))
print('Random Forest RMSE:', np.sqrt(mean_squared_error(y_test, forest.predict(X_test))))


## 9. Code Walkthrough
- We created a Sine wave with heavy noise.
- The single `DecisionTreeRegressor` got a higher RMSE because it memorized the noise on the training set.
- The `RandomForestRegressor` combined 100 trees, smoothing out the noise and generalizing better to the unseen test set.


## 10. Experiment
Let's visualize exactly *why* the Random Forest scored better.


In [ ]:
X_plot = np.linspace(0, 5, 500).reshape(-1, 1)

plt.figure(figsize=(12, 5))
plt.scatter(X, y, color='blue', alpha=0.4, label='Data')
plt.plot(X_plot, tree.predict(X_plot), color='red', alpha=0.7, label='Single Tree (Overfit)')
plt.plot(X_plot, forest.predict(X_plot), color='green', linewidth=3, label='Random Forest (Smoothed)')
plt.legend()
plt.title('Decision Tree vs Random Forest')
plt.show()


> Notice the red line zigs and zags violently. The green line is much smoother! By averaging 100 jagged trees, the Random Forest creates a beautiful, smooth approximation of the true Sine wave.


## 11. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
forest_10 = RandomForestRegressor(n_estimators=10, random_state=42)
forest_1000 = RandomForestRegressor(n_estimators=1000, random_state=42)


> **Question:** Which model will take longer to train? Does increasing `n_estimators` from 10 to 1,000 increase the risk of overfitting?

**Think before running the next cell!**


In [ ]:
print('The 1,000 tree model takes 100x longer to train.')
print('However, unlike max_depth, increasing n_estimators ALMOST NEVER causes overfitting! Averaging more trees just makes the prediction strictly more stable.')


## 12. Coding Exercise
Train a `RandomForestRegressor` but this time constrain the depth of the 100 trees inside it by passing `max_depth=2`. Plot it against `X_plot`.


In [ ]:
# YOUR CODE HERE
forest_constrained = RandomForestRegressor(n_estimators=100, max_depth=2, random_state=42)
forest_constrained.fit(X_train, y_train)

plt.scatter(X, y, color='blue', alpha=0.4)
plt.plot(X_plot, forest_constrained.predict(X_plot), color='orange', linewidth=3)
plt.title('Random Forest (max_depth=2)')
plt.show()
print('Notice how constraining the depth of the trees inside the forest creates an extremely smooth, conservative staircase!')


## 13. Debugging Challenge
A Junior Developer tried to use Random Forest to predict next month's sales, which are expected to be \$200,000. The highest historical month in the training data was \$150,000. 
The Random Forest keeps predicting \$150,000 and the developer thinks there is a bug in Scikit-learn. What is actually happening?


In [ ]:
try:
    historical_sales = np.array([[1], [2], [3], [4], [5]])
    historical_revenue = np.array([50, 75, 100, 125, 150])
    
    rf = RandomForestRegressor(n_estimators=10, random_state=42)
    rf.fit(historical_sales, historical_revenue)
    
    future_month = np.array([[6]])
    print('Predicted Revenue for Month 6:', rf.predict(future_month))
except Exception as e:
    print('Error:', e)


> **Hint:** Remember Day 12. A Random Forest is just an average of Decision Trees. Can a Decision Tree predict a number higher than what it saw in the training data? (No!). 
> Random Forests CANNOT extrapolate outside the bounds of the training data. If you need extrapolation (trending upwards infinitely), you MUST use a Linear Model.


## 14. Model Evaluation (Feature Importance)
Because Random Forests randomly subset features across hundreds of trees, they naturally discover which features are useless and which are critical. You can extract this using `.feature_importances_`. This is incredibly valuable for business stakeholders who want to know *why* the model makes predictions.


## 15. Real-World Example
In e-commerce, predicting Customer Lifetime Value (CLV) is critical. The data is messy, non-linear, has massive outliers, and missing values. A standard Linear Regression would crash or produce terrible predictions. A Random Forest naturally handles the non-linear interactions, ignores the extreme outliers, and provides a stable prediction of how much a user will spend over their lifetime.


## 16. Mini Project
Train a Random Forest on a dataset with 5 features (only the first 1 matters). Use `.feature_importances_` to prove the forest figured this out.


In [ ]:
X_multi = np.random.rand(100, 5)
y_multi = 10 * X_multi[:, 0] + np.random.randn(100) # Only feature 0 matters!

rf_feat = RandomForestRegressor(n_estimators=50, random_state=42)
rf_feat.fit(X_multi, y_multi)

importances = rf_feat.feature_importances_
for i, imp in enumerate(importances):
    print(f'Feature {i} Importance: {imp:.3f}')
print('The Forest correctly identified that Feature 0 holds ~95%+ of the predictive power!')


## 17. Common Mistakes
- **Extrapolation**: Using Random Forests for time-series forecasting where the target is trending upwards into unseen numbers.
- **Not tuning `max_depth`**: While `n_estimators` won't cause overfitting, if you leave `max_depth` unconstrained, the forest can still slightly overfit on extremely noisy datasets. 
- **Worrying about scaling**: Random Forests do not need `StandardScaler`!


## 18. Interview Questions
- **Beginner**: What is the difference between a Decision Tree and a Random Forest?
- **Intermediate**: Explain the two random steps in 'Bagging' (Bootstrap Aggregating) that ensure trees in a Random Forest are diverse.
- **Advanced**: Why does increasing `n_estimators` to infinity not cause overfitting in a Random Forest?


## 19. Knowledge Check
- What property allows you to see which features were most useful? (`.feature_importances_`)
- Does a Random Forest require scaled data? (No)


## 20. Summary
- **Random Forest** = Ensemble of Decision Trees.
- Uses **Bagging** to ensure diversity (Random Rows + Random Features).
- Averaging predictions drastically reduces variance (Overfitting).
- It cannot extrapolate outside the training data range.
- Provides feature importance out of the box.


## 21. Homework
Load the `fetch_california_housing` dataset. Train a `LinearRegression` and a `RandomForestRegressor(n_estimators=50)`. Compare their Test RMSE scores. The Random Forest should obliterate the Linear Regression because house prices are highly non-linear.
